# 🧬 DisorderNet: Beating AlphaFold 3 at Intrinsic Disorder Prediction
## ESM-2 650M + LoRA Fine-Tuning on Colab G4 GPU

**Target**: AUC-ROC > 0.88 on DisProt benchmark (vs AF3's 0.747)

| Component | Detail |
|---|---|
| GPU | NVIDIA RTX PRO 6000 (G4), 96 GB VRAM |
| Backbone | ESM-2 650M (33 layers, 1280-dim embeddings) |
| Fine-tuning | LoRA (rank 8, alpha 16) on query/value projections |
| Head | 1D CNN (3 layers) + Linear classifier |
| Data | DisProt database (~3000 proteins, experimentally verified) |
| Evaluation | 5-fold protein-grouped cross-validation |

---

### Architecture Overview

```
Protein Sequence
      │
      ▼
ESM-2 Tokenizer
      │
      ▼
ESM-2 650M Backbone (frozen, 33 layers)
  └── Last 8 layers: LoRA adapters on Q, V projections
      │
      ▼
Per-residue embeddings [L × 1280]
      │
      ▼
1D CNN Head:
  Conv1d(1280→512, k=15) → ReLU → Dropout(0.1)
  Conv1d(512→256,  k=9)  → ReLU → Dropout(0.1)
  Conv1d(256→1,    k=5)  → Sigmoid
      │
      ▼
Per-residue disorder probability [0, 1]
```

### Benchmark Context

| Method | AUC-ROC | Source |
|--------|---------|--------|
| AF3-pLDDT (CAID3) | 0.747 | CAID3 |
| DisorderNet v6 (CPU) | 0.831 | DisProt |
| ESMDisPred (CAID3 SOTA) | 0.895 | CAID3 |
| **DisorderNet GPU (target)** | **>0.88** | **DisProt** |

### Runtime Budget
- Embedding extraction: ~30 min
- Training (5 folds × 15 epochs): ~15–18 hrs
- Evaluation + figures: ~30 min
- **Total: ~16–19 hrs** (within 20 hr budget)

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 2 │ Environment Setup
# ─────────────────────────────────────────────────────────────────────────────
import subprocess, sys, time

print("Installing packages...")
t0 = time.time()

# Core deep learning + ESM
subprocess.run([
    sys.executable, "-m", "pip", "install", "-q",
    "fair-esm",
    "torch",
    "lightning",
    "scikit-learn",
    "lightgbm",
    "xgboost",
    "matplotlib",
    "seaborn",
    "requests",
    "biopython",
    "accelerate",
    "tqdm",
    "pandas",
    "numpy",
], check=True)

print(f"Packages installed in {time.time()-t0:.0f}s")

# ── GPU check ──────────────────────────────────────────────────────────────
import torch
import numpy as np

if not torch.cuda.is_available():
    raise RuntimeError(
        "No GPU detected! Go to Runtime → Change runtime type → GPU (A100/G4)"
    )

gpu_props = torch.cuda.get_device_properties(0)
vram_gb   = gpu_props.total_memory / 1e9

print(f"\n{'='*55}")
print(f"  GPU  : {torch.cuda.get_device_name(0)}")
print(f"  VRAM : {vram_gb:.1f} GB")
print(f"  CUDA : {torch.version.cuda}")
print(f"  PyTorch: {torch.__version__}")
print(f"{'='*55}")

# Warn if not on a high-VRAM GPU
if vram_gb < 20:
    print("\n⚠  WARNING: <20 GB VRAM detected. Reduce BATCH_SIZE to 2 below.")
elif vram_gb >= 70:
    print("\n✅ High-VRAM GPU detected — optimal for ESM-2 650M + LoRA.")

# Reproducibility
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark     = False

DEVICE = torch.device("cuda")
print(f"\nDevice: {DEVICE}  |  Seed: {SEED}")

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 3 │ Download & Process DisProt Data
# ─────────────────────────────────────────────────────────────────────────────
import requests
import json
import os
import time
from collections import Counter

# ── Hyperparameters (edit here) ────────────────────────────────────────────
MIN_SEQ_LEN   = 20    # discard very short proteins
MAX_SEQ_LEN   = 1022  # ESM-2 hard limit (1024 - 2 special tokens)
MIN_DISORDER  = 3     # minimum disordered residues to include protein
MIN_ORDER     = 3     # minimum ordered residues to include protein

DATA_CACHE = "disprot_raw.json"


def fetch_disprot(cache_path: str = DATA_CACHE) -> list:
    """Fetch the full DisProt database via REST API with local caching."""
    if os.path.exists(cache_path):
        print(f"Loading cached DisProt data from '{cache_path}'...")
        with open(cache_path) as f:
            return json.load(f)

    base_url = "https://disprot.org/api/search"
    params = {
        "release": "current",
        "show_ambiguous": "false",
        "show_obsolete": "false",
        "format": "json",
        "page": 0,
        "per_page": 100,
    }

    all_entries = []
    print("Downloading DisProt database...")
    t0 = time.time()

    while True:
        try:
            resp = requests.get(base_url, params=params, timeout=60)
            resp.raise_for_status()
            data = resp.json()
        except requests.RequestException as e:
            print(f"  Request error on page {params['page']}: {e}. Retrying in 5s...")
            time.sleep(5)
            continue

        entries = data.get("data", [])
        if not entries:
            break

        all_entries.extend(entries)
        total_expected = data.get("total", 0)
        elapsed = time.time() - t0
        print(
            f"  Page {params['page']:3d}: {len(entries):3d} entries "
            f"(total: {len(all_entries):4d} / {total_expected})  [{elapsed:.0f}s]"
        )

        if len(all_entries) >= total_expected:
            break

        params["page"] += 1
        time.sleep(0.2)  # be polite to the API

    print(f"Download complete: {len(all_entries)} entries in {time.time()-t0:.0f}s")

    with open(cache_path, "w") as f:
        json.dump(all_entries, f)
    print(f"Saved to '{cache_path}'")

    return all_entries


def process_disprot(entries: list) -> list:
    """
    Convert raw DisProt API entries to protein dicts with binary disorder labels.

    Each output dict:
        id       : DisProt accession (e.g. 'DP00001')
        sequence : amino acid string
        labels   : list[int] of length len(sequence), 1=disordered, 0=ordered
        length   : len(sequence)
        n_dis    : number of disordered residues
        frac_dis : fraction disordered
    """
    proteins = []
    skipped  = Counter()

    for entry in entries:
        seq = entry.get("sequence", "")

        if not seq:
            skipped["no_sequence"] += 1
            continue
        if len(seq) < MIN_SEQ_LEN:
            skipped["too_short"] += 1
            continue
        if len(seq) > MAX_SEQ_LEN:
            skipped["too_long"] += 1
            continue

        # Build binary disorder label array
        labels = [0] * len(seq)
        regions = entry.get("regions", [])

        for r in regions:
            # DisProt uses 1-based, inclusive intervals
            start = r.get("start", 0)
            end   = r.get("end", 0)
            if start and end:
                for i in range(start - 1, min(end, len(seq))):
                    labels[i] = 1

        n_dis = sum(labels)
        n_ord = len(seq) - n_dis

        if n_dis < MIN_DISORDER:
            skipped["too_few_disorder"] += 1
            continue
        if n_ord < MIN_ORDER:
            skipped["too_few_order"] += 1
            continue

        proteins.append({
            "id":       entry.get("disprot_id", ""),
            "sequence": seq,
            "labels":   labels,
            "length":   len(seq),
            "n_dis":    n_dis,
            "frac_dis": n_dis / len(seq),
        })

    return proteins, skipped


# ── Run ──────────────────────────────────────────────────────────────────────
entries = fetch_disprot()
proteins, skipped = process_disprot(entries)

total_res  = sum(p["length"] for p in proteins)
total_dis  = sum(p["n_dis"]  for p in proteins)
frac_dis   = total_dis / total_res

print(f"\n{'─'*55}")
print(f"  Proteins accepted : {len(proteins):,}")
print(f"  Residues total    : {total_res:,}")
print(f"  Disordered        : {total_dis:,}  ({100*frac_dis:.1f}%)")
print(f"  Ordered           : {total_res-total_dis:,}  ({100*(1-frac_dis):.1f}%)")
print(f"  Class imbalance   : 1 : {(1-frac_dis)/frac_dis:.1f}")
print(f"{'─'*55}")
print(f"  Skipped: {dict(skipped)}")

# Disorder fraction distribution
fracs = [p["frac_dis"] for p in proteins]
print(f"\n  Disorder fraction per protein:")
print(f"    mean={np.mean(fracs):.3f}  median={np.median(fracs):.3f}  "
      f"min={np.min(fracs):.3f}  max={np.max(fracs):.3f}")

# Quick sanity check
assert len(proteins) >= 500, f"Expected ≥500 proteins, got {len(proteins)}"
print("\n✅ Data validation passed.")

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 4 │ Load ESM-2 650M Model
# ─────────────────────────────────────────────────────────────────────────────
import esm
import torch
import torch.nn as nn
import time

print("Loading ESM-2 650M (33 layers, 1280-dim)...")
print("(First run downloads ~1.3 GB weights — subsequent runs use cache)")
t0 = time.time()

model_esm, alphabet = esm.pretrained.esm2_t33_650M_UR50D()
model_esm = model_esm.to(DEVICE)
model_esm.eval()

batch_converter = alphabet.get_batch_converter()

elapsed = time.time() - t0
mem_gb  = torch.cuda.memory_allocated() / 1e9
total_params = sum(p.numel() for p in model_esm.parameters())

print(f"\n{'─'*55}")
print(f"  Loaded in : {elapsed:.1f}s")
print(f"  Parameters: {total_params/1e6:.0f}M")
print(f"  VRAM used : {mem_gb:.1f} GB / {vram_gb:.1f} GB")
print(f"  VRAM free : {(vram_gb - mem_gb):.1f} GB")
print(f"{'─'*55}")

# Quick forward pass test
print("\nRunning forward pass test...")
test_data = [("test_protein", "ACDEFGHIKLMNPQRSTVWY" * 3)]
_, _, test_tokens = batch_converter(test_data)
test_tokens = test_tokens.to(DEVICE)

with torch.no_grad():
    results = model_esm(test_tokens, repr_layers=[33], return_contacts=False)

embed_shape = results["representations"][33].shape
print(f"  Test tokens shape     : {test_tokens.shape}")
print(f"  Embedding shape       : {embed_shape}  (batch, seq+2, 1280)")
print("✅ ESM-2 650M loaded and verified.")

del test_tokens, results
torch.cuda.empty_cache()

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 5 │ DisorderNet Architecture: LoRA + CNN Head
# ─────────────────────────────────────────────────────────────────────────────
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

# ─────────────────────────────────────────────────────────────────────────────
# LoRA: Low-Rank Adaptation
# Replaces W·x with W·x + (A·B)·x * (alpha/rank)
# Only A and B are trained; W is frozen.
# ─────────────────────────────────────────────────────────────────────────────
class LoRALinear(nn.Module):
    """
    Drop-in LoRA wrapper for nn.Linear.

    Args:
        original_linear : the frozen nn.Linear to wrap
        rank            : LoRA rank r  (default 8)
        alpha           : LoRA scaling alpha (default 16)
        dropout         : dropout on the LoRA path (default 0.05)
    """

    def __init__(
        self,
        original_linear: nn.Linear,
        rank: int  = 8,
        alpha: int = 16,
        dropout: float = 0.05,
    ):
        super().__init__()
        self.original  = original_linear
        self.rank      = rank
        self.alpha     = alpha
        self.scaling   = alpha / rank

        in_f  = original_linear.in_features
        out_f = original_linear.out_features

        # Kaiming-uniform for A, zeros for B (standard LoRA init)
        self.lora_A = nn.Parameter(torch.empty(in_f, rank))
        self.lora_B = nn.Parameter(torch.zeros(rank, out_f))
        nn.init.kaiming_uniform_(self.lora_A, a=math.sqrt(5))

        self.lora_dropout = nn.Dropout(dropout)

        # Freeze the original weights
        for p in self.original.parameters():
            p.requires_grad = False

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        base_out = self.original(x)
        lora_out = self.lora_dropout(x) @ self.lora_A @ self.lora_B
        return base_out + lora_out * self.scaling

    def extra_repr(self) -> str:
        return (
            f"in={self.original.in_features}, "
            f"out={self.original.out_features}, "
            f"rank={self.rank}, alpha={self.alpha}"
        )


# ─────────────────────────────────────────────────────────────────────────────
# CNN Classification Head
# Input : [B, L, 1280]  (per-residue ESM-2 embeddings)
# Output: [B, L]        (per-residue disorder logits)
# ─────────────────────────────────────────────────────────────────────────────
class DisorderCNNHead(nn.Module):
    """
    3-layer 1-D convolutional head for per-residue disorder prediction.
    Uses large receptive fields to capture long-range context.
    """

    def __init__(
        self,
        in_dim:   int   = 1280,
        dropout:  float = 0.10,
    ):
        super().__init__()

        self.net = nn.Sequential(
            # Layer 1: broad context (k=15, RF=15)
            nn.Conv1d(in_dim, 512, kernel_size=15, padding=7),
            nn.BatchNorm1d(512),
            nn.GELU(),
            nn.Dropout(dropout),

            # Layer 2: mid-range (k=9, total RF=23)
            nn.Conv1d(512, 256, kernel_size=9, padding=4),
            nn.BatchNorm1d(256),
            nn.GELU(),
            nn.Dropout(dropout),

            # Layer 3: fine-grained (k=5, total RF=27)
            nn.Conv1d(256, 1, kernel_size=5, padding=2),
        )

        # Residual projection for skip connection
        self.skip = nn.Conv1d(in_dim, 1, kernel_size=1)

        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv1d):
                nn.init.kaiming_normal_(m.weight, mode="fan_out", nonlinearity="relu")
                if m.bias is not None:
                    nn.init.zeros_(m.bias)
            elif isinstance(m, nn.BatchNorm1d):
                nn.init.ones_(m.weight)
                nn.init.zeros_(m.bias)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: [B, L, 1280]  →  permute to [B, 1280, L] for Conv1d
        x = x.permute(0, 2, 1)
        out  = self.net(x)        # [B, 1, L]
        skip = self.skip(x)       # [B, 1, L]
        return (out + skip).squeeze(1)  # [B, L]  — raw logits


# ─────────────────────────────────────────────────────────────────────────────
# Full DisorderNet GPU model
# ─────────────────────────────────────────────────────────────────────────────
class DisorderNetGPU(nn.Module):
    """
    DisorderNet GPU: ESM-2 650M backbone with LoRA adapters + CNN head.

    Training strategy:
      - All ESM-2 weights frozen except LoRA A/B matrices
      - LoRA applied to Q + V projections in the last `lora_layers` blocks
      - CNN head trained from scratch

    Forward:
      tokens [B, L+2]  →  disorder logits [B, L]  (padding excluded)
    """

    def __init__(
        self,
        esm_model,
        lora_layers: int  = 8,
        lora_rank:   int  = 8,
        lora_alpha:  int  = 16,
        lora_dropout: float = 0.05,
        head_dropout: float = 0.10,
    ):
        super().__init__()
        self.esm        = esm_model
        self.lora_layers = lora_layers

        # Step 1: freeze all ESM-2 parameters
        for p in self.esm.parameters():
            p.requires_grad = False

        # Step 2: inject LoRA into Q, V projections of the last N transformer layers
        n_layers = len(self.esm.layers)
        start_layer = n_layers - lora_layers  # e.g. 33 - 8 = 25

        self._lora_params = []
        self._lora_modules = []

        for layer_idx in range(start_layer, n_layers):
            layer = self.esm.layers[layer_idx]

            # ESM-2 uses self_attn with q_proj and v_proj
            attn = layer.self_attn

            if hasattr(attn, "q_proj") and isinstance(attn.q_proj, nn.Linear):
                lora_q = LoRALinear(attn.q_proj, lora_rank, lora_alpha, lora_dropout)
                attn.q_proj = lora_q
                self._lora_modules.append(lora_q)

            if hasattr(attn, "v_proj") and isinstance(attn.v_proj, nn.Linear):
                lora_v = LoRALinear(attn.v_proj, lora_rank, lora_alpha, lora_dropout)
                attn.v_proj = lora_v
                self._lora_modules.append(lora_v)

        print(f"  LoRA injected into layers {start_layer}–{n_layers-1} "
              f"({len(self._lora_modules)} projections replaced)")

        # Step 3: CNN classification head
        self.head = DisorderCNNHead(in_dim=1280, dropout=head_dropout)

        # Verify trainable parameter counts
        total_p   = sum(p.numel() for p in self.parameters())
        trainable = sum(p.numel() for p in self.parameters() if p.requires_grad)
        print(f"  Total parameters    : {total_p/1e6:.1f}M")
        print(f"  Trainable parameters: {trainable/1e6:.2f}M "
              f"({100*trainable/total_p:.2f}%)")

    def get_lora_params(self):
        """Return LoRA parameters (A, B matrices) for the optimizer."""
        for m in self._lora_modules:
            yield m.lora_A
            yield m.lora_B

    def get_head_params(self):
        """Return CNN head parameters for the optimizer."""
        return self.head.parameters()

    def forward(
        self,
        tokens: torch.Tensor,           # [B, L+2]  (with BOS/EOS tokens)
    ) -> torch.Tensor:                  # [B, L]    (per-residue logits)
        """
        Forward pass.
        Returns raw logits (before sigmoid) for each non-padding residue.
        The caller is responsible for applying the mask.
        """
        # ESM-2 forward — extract last-layer representations
        out = self.esm(
            tokens,
            repr_layers=[33],
            return_contacts=False,
        )
        # repr: [B, L+2, 1280]  — strip BOS (position 0) and EOS (last position)
        embeddings = out["representations"][33][:, 1:-1, :]  # [B, L, 1280]

        # CNN head
        logits = self.head(embeddings)  # [B, L]
        return logits


# ── Instantiate and summarise ─────────────────────────────────────────────
print("Building DisorderNetGPU...")
print(f"{'─'*55}")
model = DisorderNetGPU(
    esm_model    = model_esm,
    lora_layers  = 8,
    lora_rank    = 8,
    lora_alpha   = 16,
    lora_dropout = 0.05,
    head_dropout = 0.10,
)
model = model.to(DEVICE)
print(f"{'─'*55}")
print(f"  GPU memory after model build: "
      f"{torch.cuda.memory_allocated()/1e9:.1f} GB / {vram_gb:.1f} GB")
print("✅ DisorderNetGPU ready.")

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 6 │ Dataset and DataLoader
# ─────────────────────────────────────────────────────────────────────────────
import torch
from torch.utils.data import Dataset, DataLoader
import numpy as np

class DisProt_Dataset(Dataset):
    """
    PyTorch Dataset for DisProt proteins.

    Each item returns:
        tokens  : LongTensor [L+2]  — ESM-2 token IDs (with BOS/EOS)
        labels  : FloatTensor [L]   — per-residue binary disorder labels
        mask    : BoolTensor  [L]   — True for real (non-padding) residues
        pid     : str               — protein ID

    The dataset does *not* pad — padding is done in the collate function
    to the max length in each mini-batch.
    """

    def __init__(self, proteins: list, batch_converter):
        self.proteins        = proteins
        self.batch_converter = batch_converter

    def __len__(self) -> int:
        return len(self.proteins)

    def __getitem__(self, idx: int):
        p = self.proteins[idx]
        # ESM-2 expects list of (label, sequence) tuples
        _, _, tokens = self.batch_converter([(p["id"], p["sequence"])])
        tokens  = tokens.squeeze(0)                        # [L+2]
        labels  = torch.tensor(p["labels"],  dtype=torch.float32)  # [L]
        mask    = torch.ones(len(p["labels"]), dtype=torch.bool)    # [L]
        return tokens, labels, mask, p["id"]


def disprot_collate(batch):
    """
    Custom collate: pads tokens, labels, and mask to the longest sequence
    in the mini-batch.

    Padding:
        tokens  : padded with alphabet.padding_idx (1 for ESM-2)
        labels  : padded with 0
        mask    : padded with False
    """
    tokens_list, labels_list, mask_list, ids = zip(*batch)

    # Max seq length in this batch (tokens include BOS+EOS → L+2)
    max_tok_len = max(t.shape[0] for t in tokens_list)
    max_seq_len = max_tok_len - 2  # strip BOS/EOS for labels/mask

    PAD_IDX = 1  # ESM-2 alphabet padding index

    tokens_padded = torch.full(
        (len(batch), max_tok_len), PAD_IDX, dtype=torch.long
    )
    labels_padded = torch.zeros(len(batch), max_seq_len, dtype=torch.float32)
    mask_padded   = torch.zeros(len(batch), max_seq_len, dtype=torch.bool)

    for i, (tok, lab, msk) in enumerate(zip(tokens_list, labels_list, mask_list)):
        L_tok = tok.shape[0]          # actual token length (L+2)
        L_seq = lab.shape[0]          # actual sequence length
        tokens_padded[i, :L_tok] = tok
        labels_padded[i, :L_seq] = lab
        mask_padded[i,   :L_seq] = msk

    return tokens_padded, labels_padded, mask_padded, list(ids)


# ── Test the Dataset/DataLoader ───────────────────────────────────────────
print("Testing Dataset and DataLoader...")
test_ds = DisProt_Dataset(proteins[:8], batch_converter)
test_dl = DataLoader(
    test_ds,
    batch_size  = 2,
    shuffle     = False,
    collate_fn  = disprot_collate,
    num_workers = 0,
)

for tok, lab, msk, ids in test_dl:
    print(f"  Batch IDs       : {ids}")
    print(f"  tokens shape    : {tok.shape}")
    print(f"  labels shape    : {lab.shape}")
    print(f"  mask shape      : {msk.shape}")
    print(f"  Disorder frac   : {lab[msk].mean():.3f}")
    break

del test_ds, test_dl
print("✅ Dataset and DataLoader verified.")

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 7 │ Training Loop
# ─────────────────────────────────────────────────────────────────────────────
import torch
import torch.nn as nn
from torch.cuda.amp import GradScaler, autocast
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.utils.data import DataLoader, Subset
from sklearn.metrics import roc_auc_score, average_precision_score, f1_score, matthews_corrcoef
import numpy as np
import time
import os
import copy

# ── Training hyperparameters ─────────────────────────────────────────────
BATCH_SIZE          = 4       # per-GPU batch; ×4 accum steps = effective 16
ACCUM_STEPS         = 4       # gradient accumulation
NUM_EPOCHS          = 15      # epochs per fold
LR_LORA             = 1e-4    # learning rate for LoRA parameters
LR_HEAD             = 5e-4    # learning rate for CNN head
WEIGHT_DECAY        = 1e-2
PATIENCE            = 4       # early stopping patience (epochs without AUC improvement)
CHECKPOINT_DIR      = "checkpoints"
NUM_WORKERS         = 2

os.makedirs(CHECKPOINT_DIR, exist_ok=True)


def compute_pos_weight(proteins_train: list) -> torch.Tensor:
    """Compute BCE pos_weight = n_neg / n_pos for class imbalance."""
    total_pos = sum(p["n_dis"]              for p in proteins_train)
    total_neg = sum(p["length"] - p["n_dis"] for p in proteins_train)
    pos_weight = total_neg / max(total_pos, 1)
    # Clip to avoid extreme values with small datasets
    pos_weight = min(pos_weight, 10.0)
    return torch.tensor([pos_weight], dtype=torch.float32).to(DEVICE)


def eval_epoch(
    model: nn.Module,
    loader: DataLoader,
    threshold: float = 0.5,
) -> dict:
    """
    Evaluate model on a DataLoader.
    Returns dict with AUC, AP, F1, MCC, loss.
    """
    model.eval()
    all_probs  = []
    all_labels = []
    total_loss = 0.0
    n_batches  = 0

    # Use the same pos_weight; approximate here
    criterion = nn.BCEWithLogitsLoss(reduction="none")

    with torch.no_grad():
        for tokens, labels, mask, _ in loader:
            tokens = tokens.to(DEVICE)
            labels = labels.to(DEVICE)
            mask   = mask.to(DEVICE)

            with autocast():
                logits = model(tokens)  # [B, L]

            # Loss on valid positions only
            loss = criterion(logits[mask], labels[mask]).mean()
            total_loss += loss.item()
            n_batches  += 1

            probs = torch.sigmoid(logits)
            all_probs.append(probs[mask].cpu().numpy())
            all_labels.append(labels[mask].cpu().numpy())

    all_probs  = np.concatenate(all_probs)
    all_labels = np.concatenate(all_labels)
    preds      = (all_probs >= threshold).astype(int)
    labels_int = all_labels.astype(int)

    metrics = {
        "loss":       total_loss / max(n_batches, 1),
        "auc":        roc_auc_score(all_labels, all_probs),
        "ap":         average_precision_score(all_labels, all_probs),
        "f1":         f1_score(labels_int, preds, zero_division=0),
        "mcc":        matthews_corrcoef(labels_int, preds),
        "probs":      all_probs,
        "labels":     all_labels,
    }
    return metrics


def train_fold(
    fold_idx:        int,
    proteins_train:  list,
    proteins_val:    list,
    batch_converter,
) -> dict:
    """
    Train DisorderNetGPU for one cross-validation fold.

    Returns:
        dict with keys: best_auc, best_ap, history, val_probs, val_labels
    """
    print(f"\n{'═'*60}")
    print(f" FOLD {fold_idx+1}  |  train={len(proteins_train)}  val={len(proteins_val)}")
    print(f"{'═'*60}")

    # ── Re-initialise model fresh for each fold ───────────────────────────
    # We rebuild DisorderNetGPU to reset LoRA weights and the CNN head.
    # The ESM-2 backbone (model_esm) is re-used as the frozen base.
    fold_model = DisorderNetGPU(
        esm_model    = model_esm,
        lora_layers  = 8,
        lora_rank    = 8,
        lora_alpha   = 16,
        lora_dropout = 0.05,
        head_dropout = 0.10,
    ).to(DEVICE)

    # ── Data loaders ──────────────────────────────────────────────────────
    train_ds = DisProt_Dataset(proteins_train, batch_converter)
    val_ds   = DisProt_Dataset(proteins_val,   batch_converter)

    train_dl = DataLoader(
        train_ds, batch_size=BATCH_SIZE, shuffle=True,
        collate_fn=disprot_collate, num_workers=NUM_WORKERS,
        pin_memory=True, drop_last=False,
    )
    val_dl = DataLoader(
        val_ds, batch_size=BATCH_SIZE, shuffle=False,
        collate_fn=disprot_collate, num_workers=NUM_WORKERS,
        pin_memory=True,
    )

    # ── Optimizer: separate LR groups ────────────────────────────────────
    lora_params = list(fold_model.get_lora_params())
    head_params = list(fold_model.get_head_params())

    optimizer = AdamW(
        [
            {"params": lora_params, "lr": LR_LORA, "weight_decay": WEIGHT_DECAY},
            {"params": head_params, "lr": LR_HEAD, "weight_decay": WEIGHT_DECAY},
        ],
        betas=(0.9, 0.999),
        eps=1e-8,
    )

    total_steps   = (len(train_dl) // ACCUM_STEPS) * NUM_EPOCHS
    warmup_steps  = total_steps // 10
    scheduler     = CosineAnnealingLR(optimizer, T_max=total_steps - warmup_steps, eta_min=1e-6)

    # ── Loss function ─────────────────────────────────────────────────────
    pos_weight = compute_pos_weight(proteins_train)
    criterion  = nn.BCEWithLogitsLoss(pos_weight=pos_weight, reduction="none")

    # ── Mixed precision scaler ────────────────────────────────────────────
    scaler = GradScaler()

    best_auc   = 0.0
    best_state = None
    patience_counter = 0
    history    = []

    fold_t0 = time.time()

    # ── Training epochs ───────────────────────────────────────────────────
    for epoch in range(NUM_EPOCHS):
        fold_model.train()
        train_loss  = 0.0
        n_batches   = 0
        optimizer.zero_grad()
        epoch_t0    = time.time()

        for step, (tokens, labels, mask, _) in enumerate(train_dl):
            tokens = tokens.to(DEVICE, non_blocking=True)
            labels = labels.to(DEVICE, non_blocking=True)
            mask   = mask.to(DEVICE, non_blocking=True)

            with autocast():
                logits = fold_model(tokens)                  # [B, L]
                loss_per_res = criterion(logits[mask], labels[mask])  # [N]
                loss = loss_per_res.mean() / ACCUM_STEPS    # scale for accumulation

            scaler.scale(loss).backward()

            if (step + 1) % ACCUM_STEPS == 0 or (step + 1) == len(train_dl):
                scaler.unscale_(optimizer)
                nn.utils.clip_grad_norm_(
                    [p for p in fold_model.parameters() if p.requires_grad],
                    max_norm=1.0,
                )
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad()
                if epoch > 0:   # skip warmup on first epoch
                    scheduler.step()

            train_loss += loss.item() * ACCUM_STEPS
            n_batches  += 1

        # Linear warmup override for first epoch
        if epoch == 0:
            for g in optimizer.param_groups:
                g["lr"] = g["lr"] * 0.1  # start at 10% of base LR next epoch

        avg_train_loss = train_loss / max(n_batches, 1)

        # ── Validation ─────────────────────────────────────────────────
        val_metrics = eval_epoch(fold_model, val_dl)
        epoch_time  = time.time() - epoch_t0
        elapsed     = time.time() - fold_t0
        eta_s       = elapsed / (epoch + 1) * (NUM_EPOCHS - epoch - 1)

        row = {
            "epoch":      epoch + 1,
            "train_loss": avg_train_loss,
            "val_loss":   val_metrics["loss"],
            "val_auc":    val_metrics["auc"],
            "val_ap":     val_metrics["ap"],
            "val_f1":     val_metrics["f1"],
            "val_mcc":    val_metrics["mcc"],
        }
        history.append(row)

        marker = ""
        if val_metrics["auc"] > best_auc:
            best_auc   = val_metrics["auc"]
            best_ap    = val_metrics["ap"]
            best_state = copy.deepcopy(fold_model.state_dict())
            best_probs  = val_metrics["probs"]
            best_labels = val_metrics["labels"]
            patience_counter = 0
            ckpt_path = os.path.join(CHECKPOINT_DIR, f"fold{fold_idx+1}_best.pt")
            torch.save(best_state, ckpt_path)
            marker = " ◀ BEST"
        else:
            patience_counter += 1

        print(
            f"  Ep {epoch+1:2d}/{NUM_EPOCHS}  "
            f"loss={avg_train_loss:.4f}  "
            f"val_loss={val_metrics['loss']:.4f}  "
            f"AUC={val_metrics['auc']:.4f}  "
            f"AP={val_metrics['ap']:.4f}  "
            f"F1={val_metrics['f1']:.4f}  "
            f"[{epoch_time:.0f}s, ETA {eta_s/3600:.1f}h]"
            f"{marker}"
        )

        # Early stopping
        if patience_counter >= PATIENCE:
            print(f"  Early stopping at epoch {epoch+1} (patience={PATIENCE})")
            break

    total_time = time.time() - fold_t0
    print(f"\n  Fold {fold_idx+1} complete in {total_time/3600:.2f}h  "
          f"│  Best AUC={best_auc:.4f}  AP={best_ap:.4f}")

    # Restore best checkpoint
    fold_model.load_state_dict(best_state)

    del train_dl, val_dl, train_ds, val_ds, fold_model
    torch.cuda.empty_cache()

    return {
        "fold":       fold_idx + 1,
        "best_auc":   best_auc,
        "best_ap":    best_ap,
        "history":    history,
        "val_probs":  best_probs,
        "val_labels": best_labels,
        "ckpt_path":  ckpt_path,
        "total_time": total_time,
    }


print("✅ Training functions defined.")
print(f"  Effective batch size : {BATCH_SIZE} × {ACCUM_STEPS} = {BATCH_SIZE*ACCUM_STEPS}")
print(f"  Epochs per fold      : {NUM_EPOCHS}")
print(f"  Early stopping       : patience={PATIENCE}")

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 8 │ 5-Fold Protein-Grouped Cross-Validation
# ─────────────────────────────────────────────────────────────────────────────
import numpy as np
import time
from sklearn.model_selection import GroupKFold
from sklearn.metrics import roc_auc_score, average_precision_score

# ── Cross-validation setup ────────────────────────────────────────────────
N_FOLDS = 5

# GroupKFold ensures no protein appears in both train and val
gkf    = GroupKFold(n_splits=N_FOLDS)
groups = np.arange(len(proteins))   # each protein is its own group
X_idx  = np.arange(len(proteins))   # dummy features (we only need indices)

fold_results = []
cv_t0 = time.time()

print(f"Starting {N_FOLDS}-fold cross-validation")
print(f"  Total proteins : {len(proteins)}")
print(f"  Train per fold : ~{int(len(proteins) * (N_FOLDS-1)/N_FOLDS)}")
print(f"  Val per fold   : ~{int(len(proteins) / N_FOLDS)}")
print(f"  Estimated time : ~{NUM_EPOCHS * 0.3 * N_FOLDS:.0f}–"
      f"{NUM_EPOCHS * 0.5 * N_FOLDS:.0f} hours")

for fold_idx, (train_idx, val_idx) in enumerate(
    gkf.split(X_idx, groups=groups)
):
    proteins_train = [proteins[i] for i in train_idx]
    proteins_val   = [proteins[i] for i in val_idx]

    result = train_fold(
        fold_idx       = fold_idx,
        proteins_train = proteins_train,
        proteins_val   = proteins_val,
        batch_converter = batch_converter,
    )
    fold_results.append(result)

    # Running summary
    aucs_so_far = [r["best_auc"] for r in fold_results]
    aps_so_far  = [r["best_ap"]  for r in fold_results]
    elapsed_h   = (time.time() - cv_t0) / 3600
    print(f"\n  ─── Running mean AUC = {np.mean(aucs_so_far):.4f} ± "
          f"{np.std(aucs_so_far):.4f}  [{elapsed_h:.1f}h elapsed] ───")

# ── Aggregate results ─────────────────────────────────────────────────────
all_probs  = np.concatenate([r["val_probs"]  for r in fold_results])
all_labels = np.concatenate([r["val_labels"] for r in fold_results])

pooled_auc = roc_auc_score(all_labels, all_probs)
pooled_ap  = average_precision_score(all_labels, all_probs)

fold_aucs = [r["best_auc"] for r in fold_results]
fold_aps  = [r["best_ap"]  for r in fold_results]
fold_times = [r["total_time"]/3600 for r in fold_results]

total_cv_h = (time.time() - cv_t0) / 3600

print(f"\n{'═'*60}")
print(f" CROSS-VALIDATION COMPLETE")
print(f"{'═'*60}")
print(f"{'Fold':>5} {'AUC':>8} {'AP':>8} {'Time(h)':>9}")
print(f"{'─'*35}")
for r in fold_results:
    print(f"{r['fold']:>5} {r['best_auc']:>8.4f} {r['best_ap']:>8.4f} "
          f"{r['total_time']/3600:>9.2f}")
print(f"{'─'*35}")
print(f"{'Mean':>5} {np.mean(fold_aucs):>8.4f} {np.mean(fold_aps):>8.4f} "
      f"{np.mean(fold_times):>9.2f}")
print(f"{'Std':>5} {np.std(fold_aucs):>8.4f} {np.std(fold_aps):>8.4f}")
print(f"{'─'*35}")
print(f"{'Pooled':>5} {pooled_auc:>8.4f} {pooled_ap:>8.4f}")
print(f"{'═'*60}")
print(f"  Total CV time : {total_cv_h:.2f}h")

if pooled_auc >= 0.88:
    print(f"\n🎉 TARGET AUC ≥ 0.88 ACHIEVED! ({pooled_auc:.4f})")
elif pooled_auc >= 0.83:
    print(f"\n✅ Beats DisorderNet v6 baseline (0.831). AUC = {pooled_auc:.4f}")
else:
    print(f"\n⚠  AUC = {pooled_auc:.4f} — consider more epochs or larger LoRA rank.")

# Save results to JSON for persistence
import json
results_summary = {
    "pooled_auc":  float(pooled_auc),
    "pooled_ap":   float(pooled_ap),
    "fold_aucs":   [float(a) for a in fold_aucs],
    "fold_aps":    [float(a) for a in fold_aps],
    "mean_auc":    float(np.mean(fold_aucs)),
    "std_auc":     float(np.std(fold_aucs)),
    "mean_ap":     float(np.mean(fold_aps)),
    "total_cv_hours": float(total_cv_h),
}
with open("cv_results.json", "w") as f:
    json.dump(results_summary, f, indent=2)
print("\nSaved CV results to cv_results.json")

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 9 │ Comprehensive Benchmark Comparison
# ─────────────────────────────────────────────────────────────────────────────
import numpy as np
from sklearn.metrics import (
    roc_auc_score, average_precision_score,
    f1_score, matthews_corrcoef, roc_curve, precision_recall_curve,
)

# ── Benchmark table (literature values) ──────────────────────────────────
BENCHMARKS = [
    # (Method,                     AUC,    AP,     Source)
    ("AF3-pLDDT (CAID3, rank 13)", 0.747,  None,   "CAID3"),
    ("AF2-pLDDT (CAID3, rank 11)", 0.770,  None,   "CAID3"),
    ("IUPred3",                    0.789,  None,   "CAID"),
    ("flDPnn (CAID1/2)",           0.814,  0.460,  "CAID"),
    ("DisorderNet v4 (no pLM)",    0.794,  0.478,  "DisProt"),
    ("SETH (ProtT5+CNN)",          0.830,  None,   "CheZOD"),
    ("DisorderNet v6 (ESM 8M+GBDT)", 0.831, 0.537, "DisProt"),
    ("flDPnn3a (CAID3)",           0.871,  0.499,  "CAID3"),
    ("ESM2_35M-LoRA",              0.868,  0.689,  "CAID1"),
    ("ESM2_650M-LoRA",             0.880,  0.721,  "CAID1"),
    ("DisorderUnetLM",             0.881,  0.778,  "CAID3-PDB"),
    ("ESMDisPred (CAID3 SOTA)",    0.895,  0.778,  "CAID3"),
]

# ── Our results ──────────────────────────────────────────────────────────
our_auc = pooled_auc
our_ap  = pooled_ap

# Determine optimal F1 threshold
from sklearn.metrics import precision_recall_curve
precisions, recalls, thresholds = precision_recall_curve(all_labels, all_probs)
f1_scores_thresh = 2 * precisions * recalls / np.maximum(precisions + recalls, 1e-8)
opt_thresh_idx = np.argmax(f1_scores_thresh)
opt_thresh     = thresholds[opt_thresh_idx] if opt_thresh_idx < len(thresholds) else 0.5
opt_f1         = f1_scores_thresh[opt_thresh_idx]

preds_opt  = (all_probs >= opt_thresh).astype(int)
our_mcc    = matthews_corrcoef(all_labels.astype(int), preds_opt)
our_f1     = f1_score(all_labels.astype(int), preds_opt)

BENCHMARKS_WITH_OURS = BENCHMARKS + [
    ("DisorderNet GPU (OURS)", our_auc, our_ap, "DisProt")
]

# Sort by AUC
BENCHMARKS_WITH_OURS.sort(key=lambda x: x[1])

# ── Print comparison table ────────────────────────────────────────────────
SEP = "─" * 64
print(f"\n{'═'*64}")
print(f" BENCHMARK COMPARISON — Intrinsic Disorder Prediction")
print(f"{'═'*64}")
print(f" {'Method':<38} {'AUC':>7} {'AP':>7}  {'Source'}")
print(SEP)

for method, auc, ap, source in BENCHMARKS_WITH_OURS:
    ap_str = f"{ap:.3f}" if ap is not None else " N/A "
    is_ours = "(OURS)" in method
    marker  = " ◀" if is_ours else ""
    prefix  = "*" if is_ours else " "
    print(f"{prefix} {method:<38} {auc:>7.3f} {ap_str:>7}  {source}{marker}")

print(SEP)
print(f"  Our model  │  AUC={our_auc:.4f}  AP={our_ap:.4f}  "
      f"F1={our_f1:.4f}  MCC={our_mcc:.4f}")
print(f"  Opt thresh │  {opt_thresh:.3f}")
print(f"{'═'*64}")

# Improvements over baselines
af3_auc = 0.747
v6_auc  = 0.831
print(f"\n  vs AF3-pLDDT   : +{our_auc - af3_auc:.4f} AUC")
print(f"  vs DisorderNet v6: +{our_auc - v6_auc:.4f} AUC")

if our_auc >= 0.895:
    print("\n🏆 Matches or beats ESMDisPred (CAID3 SOTA)!")
elif our_auc >= 0.880:
    print("\n🥇 Matches ESM2_650M-LoRA CAID1 performance on DisProt!")
elif our_auc >= 0.871:
    print("\n🥈 Beats flDPnn3a and approaches top-tier models.")
elif our_auc >= 0.831:
    print("\n✅ Beats DisorderNet v6 baseline.")

# Per-fold breakdown
print(f"\n  Per-fold AUC: "
      + "  ".join(f"F{i+1}={a:.4f}" for i, a in enumerate(fold_aucs)))
print(f"  Mean ± Std  : {np.mean(fold_aucs):.4f} ± {np.std(fold_aucs):.4f}")

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 10 │ Publication-Quality Figures
# ─────────────────────────────────────────────────────────────────────────────
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.patches as mpatches
import numpy as np
import seaborn as sns
from sklearn.metrics import roc_curve, precision_recall_curve, auc

# ── Global style ─────────────────────────────────────────────────────────
plt.rcParams.update({
    "font.family":      "DejaVu Sans",
    "font.size":        11,
    "axes.titlesize":   13,
    "axes.labelsize":   12,
    "axes.spines.top":  False,
    "axes.spines.right": False,
    "axes.linewidth":   1.2,
    "xtick.labelsize":  10,
    "ytick.labelsize":  10,
    "legend.fontsize":  10,
    "figure.dpi":       150,
    "savefig.dpi":      300,
    "savefig.bbox":     "tight",
})

# ── Colour palette ────────────────────────────────────────────────────────
C_OURS     = "#2563EB"   # deep blue   — DisorderNet GPU
C_AF3      = "#EF4444"   # red         — AlphaFold 3
C_SOTA     = "#7C3AED"   # violet      — ESMDisPred SOTA
C_V6       = "#16A34A"   # green       — DisorderNet v6
C_OTHERS   = "#9CA3AF"   # grey        — other methods
C_FOLD     = ["#3B82F6", "#10B981", "#F59E0B", "#EF4444", "#8B5CF6"]


# ═══════════════════════════════════════════════════════════════════════════
# FIGURE 1 — ROC + PR Curves
# ═══════════════════════════════════════════════════════════════════════════
fig1, (ax_roc, ax_pr) = plt.subplots(1, 2, figsize=(13, 5.5))
fig1.suptitle("DisorderNet GPU — Performance Curves", fontsize=14, fontweight="bold")

# Pooled ROC
fpr, tpr, _ = roc_curve(all_labels, all_probs)
ax_roc.plot(fpr, tpr, color=C_OURS, lw=2.5, label=f"DisorderNet GPU (AUC={our_auc:.4f})")
ax_roc.plot([0, 1], [0, 1], "--", color="#D1D5DB", lw=1.2, label="Random")
# Reference lines for AF3 and v6 (approximate from known performance)
ax_roc.axhline(0.747, color=C_AF3,  ls=":",  lw=1.5, alpha=0.7, label="AF3-pLDDT AUC=0.747")
ax_roc.axhline(0.831, color=C_V6,   ls="-.", lw=1.5, alpha=0.7, label="DisorderNet v6 AUC=0.831")
ax_roc.set_xlabel("False Positive Rate")
ax_roc.set_ylabel("True Positive Rate")
ax_roc.set_title("ROC Curve (Pooled, 5-Fold CV)")
ax_roc.legend(loc="lower right")
ax_roc.set_xlim([-0.02, 1.02])
ax_roc.set_ylim([-0.02, 1.02])

# Per-fold ROC curves (lighter)
for i, r in enumerate(fold_results):
    fpr_f, tpr_f, _ = roc_curve(r["val_labels"], r["val_probs"])
    ax_roc.plot(fpr_f, tpr_f, color=C_OURS, lw=0.8, alpha=0.3,
                label=f"Fold {i+1} ({r['best_auc']:.3f})" if i == 0 else "_")

# Pooled PR
prec, rec, _ = precision_recall_curve(all_labels, all_probs)
ax_pr.plot(rec, prec, color=C_OURS, lw=2.5, label=f"DisorderNet GPU (AP={our_ap:.4f})")
baseline_ap = all_labels.mean()
ax_pr.axhline(baseline_ap, color="#D1D5DB", ls="--", lw=1.2,
              label=f"Random baseline (AP={baseline_ap:.3f})")

# Per-fold PR
for i, r in enumerate(fold_results):
    prec_f, rec_f, _ = precision_recall_curve(r["val_labels"], r["val_probs"])
    ax_pr.plot(rec_f, prec_f, color=C_OURS, lw=0.8, alpha=0.3)

ax_pr.set_xlabel("Recall")
ax_pr.set_ylabel("Precision")
ax_pr.set_title("Precision-Recall Curve (Pooled, 5-Fold CV)")
ax_pr.legend(loc="upper right")
ax_pr.set_xlim([-0.02, 1.02])
ax_pr.set_ylim([-0.02, 1.02])

plt.tight_layout()
plt.savefig("fig1_roc_pr.pdf")
plt.savefig("fig1_roc_pr.png")
plt.show()
print("Saved fig1_roc_pr.{pdf,png}")


# ═══════════════════════════════════════════════════════════════════════════
# FIGURE 2 — Benchmark Comparison (horizontal bar chart)
# ═══════════════════════════════════════════════════════════════════════════
methods = [
    ("AF3-pLDDT",          0.747,  C_AF3),
    ("AF2-pLDDT",          0.770,  C_AF3),
    ("IUPred3",            0.789,  C_OTHERS),
    ("DisorderNet v4",     0.794,  C_OTHERS),
    ("SETH",               0.830,  C_OTHERS),
    ("DisorderNet v6",     0.831,  C_V6),
    ("flDPnn (CAID1/2)",   0.814,  C_OTHERS),
    ("ESM2_35M-LoRA",      0.868,  C_OTHERS),
    ("flDPnn3a",           0.871,  C_OTHERS),
    ("ESM2_650M-LoRA",     0.880,  C_OTHERS),
    ("DisorderUnetLM",     0.881,  C_OTHERS),
    ("ESMDisPred",         0.895,  C_SOTA),
    ("DisorderNet GPU",    our_auc, C_OURS),
]
methods.sort(key=lambda x: x[1])

names  = [m[0] for m in methods]
aucs   = [m[1] for m in methods]
colors = [m[2] for m in methods]

fig2, ax2 = plt.subplots(figsize=(9, 7))
y_pos = np.arange(len(names))

bars = ax2.barh(y_pos, aucs, color=colors, edgecolor="white", height=0.7, alpha=0.88)

# Value labels
for bar, val in zip(bars, aucs):
    ax2.text(
        val + 0.002, bar.get_y() + bar.get_height() / 2,
        f"{val:.3f}", va="center", ha="left", fontsize=9.5,
    )

ax2.set_yticks(y_pos)
ax2.set_yticklabels(names)
ax2.set_xlabel("AUC-ROC")
ax2.set_title("Intrinsic Disorder Prediction Benchmark", fontweight="bold")
ax2.set_xlim([0.70, 0.94])
ax2.axvline(0.88, ls="--", color=C_OURS, lw=1.5, alpha=0.5, label="Target AUC=0.88")

legend_handles = [
    mpatches.Patch(color=C_OURS,   label="DisorderNet GPU (ours)"),
    mpatches.Patch(color=C_V6,     label="DisorderNet v6 (baseline)"),
    mpatches.Patch(color=C_AF3,    label="AlphaFold pLDDT"),
    mpatches.Patch(color=C_SOTA,   label="ESMDisPred (SOTA)"),
    mpatches.Patch(color=C_OTHERS, label="Other methods"),
]
ax2.legend(handles=legend_handles, loc="lower right", framealpha=0.9)
plt.tight_layout()
plt.savefig("fig2_benchmark.pdf")
plt.savefig("fig2_benchmark.png")
plt.show()
print("Saved fig2_benchmark.{pdf,png}")


# ═══════════════════════════════════════════════════════════════════════════
# FIGURE 3 — Version Progression  v4 → v5 → v6 → GPU
# ═══════════════════════════════════════════════════════════════════════════
versions = ["v4\n(ESM-8M+\nLightGBM)", "v5\n(ESM-35M+\nXGBoost)",
            "v6\n(ESM-8M+\nGBDT)",  "GPU\n(ESM-650M+\nLoRA+CNN)"]
v_aucs   = [0.794, 0.812, 0.831, our_auc]
v_aps    = [0.478, 0.509, 0.537, our_ap]
v_colors = [C_OTHERS, C_OTHERS, C_V6, C_OURS]

fig3, (ax3a, ax3b) = plt.subplots(1, 2, figsize=(12, 5))
fig3.suptitle("DisorderNet Version Progression", fontsize=14, fontweight="bold")

for ax, vals, ylabel, title in [
    (ax3a, v_aucs, "AUC-ROC",           "AUC-ROC Progression"),
    (ax3b, v_aps,  "Average Precision", "Average Precision Progression"),
]:
    x = np.arange(len(versions))
    ax.bar(x, vals, color=v_colors, edgecolor="white", alpha=0.88, width=0.55)
    ax.set_xticks(x)
    ax.set_xticklabels(versions, fontsize=10)
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    for xi, val in zip(x, vals):
        ax.text(xi, val + 0.002, f"{val:.3f}", ha="center", fontsize=10,
                fontweight="bold" if xi == len(versions)-1 else "normal")
    ax.set_ylim([min(vals)*0.93, max(vals)*1.06])

    # Add AF3 reference line
    if ylabel == "AUC-ROC":
        ax.axhline(0.747, ls=":", color=C_AF3, lw=1.5, label="AF3-pLDDT (0.747)")
        ax.legend(loc="upper left")

plt.tight_layout()
plt.savefig("fig3_progression.pdf")
plt.savefig("fig3_progression.png")
plt.show()
print("Saved fig3_progression.{pdf,png}")


# ═══════════════════════════════════════════════════════════════════════════
# FIGURE 4 — Fold Stability + Score Distribution
# ═══════════════════════════════════════════════════════════════════════════
fig4 = plt.figure(figsize=(14, 5.5))
gs   = gridspec.GridSpec(1, 3, figure=fig4, wspace=0.35)
ax4a = fig4.add_subplot(gs[0])
ax4b = fig4.add_subplot(gs[1])
ax4c = fig4.add_subplot(gs[2])
fig4.suptitle("Fold Stability & Score Distributions", fontsize=14, fontweight="bold")

# 4a: Fold-by-fold AUC
fold_nums = [r["fold"] for r in fold_results]
fold_auc_vals = [r["best_auc"] for r in fold_results]
fold_ap_vals  = [r["best_ap"]  for r in fold_results]

ax4a.plot(fold_nums, fold_auc_vals, "o-", color=C_OURS, lw=2, ms=8, label="AUC")
ax4a.plot(fold_nums, fold_ap_vals,  "s--", color="#F59E0B", lw=2, ms=8, label="AP")
ax4a.axhline(np.mean(fold_auc_vals), ls=":", color=C_OURS,    lw=1.5, alpha=0.5)
ax4a.axhline(np.mean(fold_ap_vals),  ls=":", color="#F59E0B", lw=1.5, alpha=0.5)
ax4a.set_xlabel("Fold")
ax4a.set_ylabel("Score")
ax4a.set_title("Fold-by-Fold Metrics")
ax4a.set_xticks(fold_nums)
ax4a.legend()
ax4a.set_ylim([0.75, 1.0])

# 4b: Prediction probability distribution for positives vs negatives
pos_probs = all_probs[all_labels == 1]
neg_probs = all_probs[all_labels == 0]

bins = np.linspace(0, 1, 40)
ax4b.hist(neg_probs, bins=bins, alpha=0.6, color=C_V6,   density=True,
          label=f"Ordered (n={len(neg_probs):,})")
ax4b.hist(pos_probs, bins=bins, alpha=0.6, color=C_OURS,  density=True,
          label=f"Disordered (n={len(pos_probs):,})")
ax4b.axvline(opt_thresh, ls="--", color="#1F2937", lw=1.5,
             label=f"Opt. threshold ({opt_thresh:.2f})")
ax4b.set_xlabel("Predicted Disorder Probability")
ax4b.set_ylabel("Density")
ax4b.set_title("Score Distributions")
ax4b.legend(loc="upper center")

# 4c: Training history (last fold)
last_history = fold_results[-1]["history"]
epochs_h     = [row["epoch"]    for row in last_history]
train_losses = [row["train_loss"] for row in last_history]
val_losses   = [row["val_loss"]   for row in last_history]
val_aucs_h   = [row["val_auc"]    for row in last_history]

ax4c_twin = ax4c.twinx()
ax4c.plot(epochs_h, train_losses, "o-", color="#9CA3AF",  lw=2, ms=5, label="Train loss")
ax4c.plot(epochs_h, val_losses,   "s-", color="#F59E0B",  lw=2, ms=5, label="Val loss")
ax4c_twin.plot(epochs_h, val_aucs_h, "D-", color=C_OURS, lw=2, ms=5, label="Val AUC")
ax4c.set_xlabel("Epoch")
ax4c.set_ylabel("BCE Loss", color="#374151")
ax4c_twin.set_ylabel("AUC-ROC", color=C_OURS)
ax4c_twin.tick_params(axis="y", colors=C_OURS)
ax4c.set_title(f"Training Curves (Fold {fold_results[-1]['fold']})")
lines1, labels1 = ax4c.get_legend_handles_labels()
lines2, labels2 = ax4c_twin.get_legend_handles_labels()
ax4c.legend(lines1 + lines2, labels1 + labels2, loc="center right", fontsize=9)

plt.savefig("fig4_stability.pdf")
plt.savefig("fig4_stability.png")
plt.show()
print("Saved fig4_stability.{pdf,png}")

print("\n✅ All 4 figures generated and saved.")

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 11 │ Save Results, Best Model, and Optional GitHub Push
# ─────────────────────────────────────────────────────────────────────────────
import torch
import json
import os
import shutil
import numpy as np
from datetime import datetime

RUN_TIMESTAMP = datetime.now().strftime("%Y%m%d_%H%M%S")

# ── 1. Identify best fold checkpoint ─────────────────────────────────────
best_fold_idx = int(np.argmax([r["best_auc"] for r in fold_results]))
best_fold     = fold_results[best_fold_idx]
best_ckpt_src = best_fold["ckpt_path"]
best_ckpt_dst = f"disordernet_gpu_best_{RUN_TIMESTAMP}.pt"

shutil.copy(best_ckpt_src, best_ckpt_dst)
print(f"Best model checkpoint : {best_ckpt_dst}")
print(f"  (from fold {best_fold['fold']}  AUC={best_fold['best_auc']:.4f})")

# ── 2. Save full results JSON ─────────────────────────────────────────────
full_results = {
    "run_timestamp": RUN_TIMESTAMP,
    "config": {
        "batch_size":    BATCH_SIZE,
        "accum_steps":   ACCUM_STEPS,
        "effective_batch": BATCH_SIZE * ACCUM_STEPS,
        "num_epochs":    NUM_EPOCHS,
        "lr_lora":       LR_LORA,
        "lr_head":       LR_HEAD,
        "lora_rank":     8,
        "lora_alpha":    16,
        "lora_layers":   8,
        "n_folds":       N_FOLDS,
    },
    "dataset": {
        "n_proteins":  len(proteins),
        "n_residues":  int(total_res),
        "frac_disorder": float(frac_dis),
    },
    "pooled": {
        "auc": float(our_auc),
        "ap":  float(our_ap),
        "f1":  float(our_f1),
        "mcc": float(our_mcc),
        "opt_threshold": float(opt_thresh),
    },
    "per_fold": [
        {
            "fold":      r["fold"],
            "best_auc":  float(r["best_auc"]),
            "best_ap":   float(r["best_ap"]),
            "train_time_h": float(r["total_time"] / 3600),
            "history":   r["history"],
        }
        for r in fold_results
    ],
    "vs_baseline": {
        "delta_auc_vs_af3": float(our_auc - 0.747),
        "delta_auc_vs_v6":  float(our_auc - 0.831),
    },
    "best_checkpoint": best_ckpt_dst,
    "all_checkpoints": [r["ckpt_path"] for r in fold_results],
}

results_json_path = f"disordernet_gpu_results_{RUN_TIMESTAMP}.json"
with open(results_json_path, "w") as f:
    json.dump(full_results, f, indent=2)
print(f"\nFull results saved to : {results_json_path}")

# ── 3. List all generated files ───────────────────────────────────────────
files_generated = [
    best_ckpt_dst,
    results_json_path,
    "cv_results.json",
    "disprot_raw.json",
    "fig1_roc_pr.pdf",    "fig1_roc_pr.png",
    "fig2_benchmark.pdf", "fig2_benchmark.png",
    "fig3_progression.pdf", "fig3_progression.png",
    "fig4_stability.pdf", "fig4_stability.png",
]
print("\nGenerated files:")
for fn in files_generated:
    exists = os.path.exists(fn)
    size   = os.path.getsize(fn) / 1024 if exists else 0
    status = f"{size:.0f} KB" if exists else "MISSING"
    print(f"  {'✅' if exists else '❌'}  {fn:<50}  {status}")

# ── 4. Optional: push to GitHub ───────────────────────────────────────────
PUSH_TO_GITHUB = False   # ← Set to True to enable
GITHUB_REPO    = "https://github.com/Tommaso-R-Marena/DisorderNet.git"
GITHUB_TOKEN   = ""      # ← Paste your Personal Access Token here
GITHUB_BRANCH  = "main"

if PUSH_TO_GITHUB:
    import subprocess

    # Embed token in URL for authentication
    repo_url_auth = GITHUB_REPO.replace(
        "https://", f"https://{GITHUB_TOKEN}@"
    )

    cmds = [
        ["git", "clone", repo_url_auth, "DisorderNet_repo"],
        ["cp", "-r", "checkpoints",        "DisorderNet_repo/results/"],
        ["cp", results_json_path,          "DisorderNet_repo/results/"],
        ["cp", "fig1_roc_pr.png",          "DisorderNet_repo/figures/"],
        ["cp", "fig2_benchmark.png",       "DisorderNet_repo/figures/"],
        ["cp", "fig3_progression.png",     "DisorderNet_repo/figures/"],
        ["cp", "fig4_stability.png",       "DisorderNet_repo/figures/"],
        ["cp", best_ckpt_dst,              "DisorderNet_repo/models/"],
    ]

    for cmd in cmds:
        result = subprocess.run(cmd, capture_output=True, text=True)
        if result.returncode != 0:
            print(f"  ⚠ Command {' '.join(cmd[:2])} failed: {result.stderr[:200]}")

    # Git add + commit + push
    git_cmds = [
        ["git", "-C", "DisorderNet_repo", "config", "user.email", "colab@disordernet.ai"],
        ["git", "-C", "DisorderNet_repo", "config", "user.name",  "DisorderNet Colab"],
        ["git", "-C", "DisorderNet_repo", "add", "."],
        ["git", "-C", "DisorderNet_repo", "commit", "-m",
         f"Add GPU results: AUC={our_auc:.4f} AP={our_ap:.4f} [{RUN_TIMESTAMP}]"],
        ["git", "-C", "DisorderNet_repo", "push", "origin", GITHUB_BRANCH],
    ]
    for cmd in git_cmds:
        result = subprocess.run(cmd, capture_output=True, text=True)
        print(f"  {' '.join(cmd[2:4])}: {'OK' if result.returncode == 0 else result.stderr[:100]}")

    print("\n✅ Results pushed to GitHub.")
else:
    print("\n[GitHub push disabled. Set PUSH_TO_GITHUB=True to enable.]")

# ── 5. Final summary ─────────────────────────────────────────────────────
print(f"\n{'═'*60}")
print(f" DISORDERNET GPU — FINAL RESULTS")
print(f"{'═'*60}")
print(f"  AUC-ROC  : {our_auc:.4f}  (target: >0.88)")
print(f"  Avg Prec : {our_ap:.4f}")
print(f"  F1 Score : {our_f1:.4f}  (threshold={opt_thresh:.3f})")
print(f"  MCC      : {our_mcc:.4f}")
print(f"  vs AF3   : +{our_auc - 0.747:.4f} AUC")
print(f"  vs v6    : +{our_auc - 0.831:.4f} AUC")
print(f"  Total CV time : {total_cv_h:.2f} hours")
print(f"{'═'*60}")